In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
try:
    from lightgbm import LGBMRegressor
except OSError as e:
    if "libomp" in str(e).lower():
        raise OSError(
            "LightGBM could not load libomp (OpenMP).\n"
            "macOS + Homebrew fix:\n"
            "  brew install libomp\n"
            "Then restart the kernel.\n"
            "Conda fix:\n"
            "  conda install -c conda-forge llvm-openmp\n"
        ) from e
    raise

BASE_DIR = Path("/Users/vishalsankarram/dsci558-project/game_feature_export/artifacts/no_sentiment_with_value")
parquet_path = BASE_DIR / "cleaned_data.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Missing file: {parquet_path}")

In [4]:
df = pd.read_parquet(parquet_path)
print("Original shape:", df.shape)
print(df.head())

Original shape: (21690, 350)
   bgg_id                                     mean_embedding  \
0       1  [-0.03231392055749893, 0.030320700258016586, 0...   
1      10  [-0.007255558390170336, 0.03381650522351265, -...   
2  100046  [-0.13280192017555237, -0.06126563996076584, 0...   
3   10007  [-0.03585081920027733, 0.03831823170185089, -0...   
4  100172  [-0.027283532544970512, 0.03955841809511185, -...   

                               description_embedding  n_reviews  \
0  [0.016889318823814392, 0.0013508893316611648, ...   4.000000   
1  [0.12207939475774765, 0.027984006330370903, -0...   4.000000   
2  [-0.02486669272184372, -0.02881644479930401, -...  -0.307692   
3  [-0.09014654904603958, -0.019658751785755157, ...  -0.253846   
4  [0.007031145039945841, 0.026811618357896805, -...   0.223077   

   review_count_feature  dispersion  median_distance_to_mean  \
0              1.482218    0.055270                -0.200072   
1              1.508356    0.204980                 0.0

In [5]:
# Drop columns by prefix only
prefix_re = r"(mech_|cat_|rank)"
df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]

/var/folders/tq/jtgfkbln5bv4lmlbq2tvnq1h0000gn/T/ipykernel_33478/1376397886.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]


In [6]:
print(df.columns)

Index(['bgg_id', 'mean_embedding', 'description_embedding', 'n_reviews',
       'review_count_feature', 'dispersion', 'median_distance_to_mean',
       'max_distance_to_mean', 'n_unique_reviewers', 'reviews_per_user_mean',
       'reviews_per_user_std', 'value_score_mean', 'value_score_std', 'year',
       'geek_rating', 'avg_rating', 'bayes_average', 'num_voters',
       'is_expansion', 'min_players', 'max_players', 'best_min_players',
       'best_max_players', 'min_playtime', 'max_playtime', 'min_age',
       'complexity', 'has_games_csv', 'coll_share_owns', 'coll_share_wants',
       'coll_share_wtb', 'coll_share_wtt', 'has_description_embedding',
       'log1p_last_mean', 'n_weeks_observed', 'price_slope_4w', 'price_vol',
       'price_coverage', 'log1p_last_min', 'log1p_last_max',
       'last_week_spread', 'log1p_mean_hist', 'log1p_median_weekly_mean',
       'mean_vs_last_ratio', 'price_slope_12w', 'price_slope_full',
       'pct_change_first_to_last', 'price_cv', 'price_iqr',


In [7]:
# Drop known unnecessary columns if present
cols_to_drop = [
    "categories", "mechanisms", "primary_category", "game_name", "mean_embedding", "description_embedding",
    "is_expansion", "has_description_embedding", "has_games_csv",
    "stage_c_split",'n_reviews', 'review_count_feature', 'dispersion',
       'median_distance_to_mean', 'max_distance_to_mean', 'n_unique_reviewers',
       'reviews_per_user_mean','reviews_per_user_std', 'value_score_mean', 'value_score_std',
       'year', 'geek_rating', 'avg_rating', 'bayes_average', 'num_voters',
       'min_players', 'max_players', 'best_min_players', 'best_max_players',
       'min_playtime', 'max_playtime', 'min_age', 'complexity',
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("After drop:", df.shape)

After drop: (21690, 28)


In [8]:
df.columns

Index(['bgg_id', 'coll_share_owns', 'coll_share_wants', 'coll_share_wtb',
       'coll_share_wtt', 'log1p_last_mean', 'n_weeks_observed',
       'price_slope_4w', 'price_vol', 'price_coverage', 'log1p_last_min',
       'log1p_last_max', 'last_week_spread', 'log1p_mean_hist',
       'log1p_median_weekly_mean', 'mean_vs_last_ratio', 'price_slope_12w',
       'price_slope_full', 'pct_change_first_to_last', 'price_cv', 'price_iqr',
       'max_drawdown_mean', 'weeks_since_last_obs', 'calendar_span_weeks',
       'trailing_null_frac_12w', 'mean_intraweek_range_last',
       'mean_intraweek_range_avg', 'stage_a_split'],
      dtype='object')

In [9]:
df.head()

,bgg_id,coll_share_owns,coll_share_wants,coll_share_wtb,coll_share_wtt,log1p_last_mean,n_weeks_observed,price_slope_4w,price_vol,price_coverage,...,pct_change_first_to_last,price_cv,price_iqr,max_drawdown_mean,weeks_since_last_obs,calendar_span_weeks,trailing_null_frac_12w,mean_intraweek_range_last,mean_intraweek_range_avg,stage_a_split
0,1,0.872523,0.050859,0.027081,0.049538,0.285581,0.228916,0.100,4.000000,0.0,...,-4.000000,4.000000,4.000000,1.884206,0.0,0.3290,0.0,4.0,0.965327,train
1,10,0.911374,0.022157,0.008124,0.058346,0.349009,-0.325301,0.000,-0.329732,-1.0,...,0.000000,-0.385510,-0.193407,-0.392238,0.0,0.3290,1.0,0.0,-0.088124,train
2,100046,NaN,NaN,NaN,NaN,-0.386078,-0.234940,-0.135,-0.286851,0.0,...,-0.995780,-0.087633,-0.193407,-0.111725,0.0,0.2625,1.0,0.0,-0.082727,train
3,10007,0.736842,0.000000,0.000000,0.263158,-0.057413,0.349398,0.000,0.133796,0.0,...,-1.992115,0.381356,0.685714,0.392015,0.0,-0.2765,0.0,0.0,-0.054926,train
4,100172,0.775510,0.040816,0.040816,0.142857,0.057999,0.789157,0.000,2.273990,0.0,...,-4.000000,1.606227,3.474725,1.508212,0.0,0.3290,1.0,0.0,0.085966,train


In [10]:
import json
import os
from collections import defaultdict

def load_jsonl_dir(directory):
    records = []
    
    for filename in os.listdir(directory):
        if filename.endswith(".jsonl"):
            path = os.path.join(directory, filename)
            
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        records.append(json.loads(line))
    
    return records

def aggregate_by_bgg(records):
    agg = defaultdict(list)
    
    for r in records:
        bgg_id = r.get("bgg_id")
        if bgg_id is not None:
            agg[bgg_id].append(r)
    
    return agg

def summarize(agg):
    summary = {}
    
    for bgg_id, items in agg.items():
        quality_scores = [x["quality_score"] for x in items if x.get("quality_score") is not None]
        
        sentiments = [x["sentiment"] for x in items if x.get("sentiment")]
        
        sentiment_counts = {
            "positive": sentiments.count("positive"),
            "neutral": sentiments.count("neutral"),
            "negative": sentiments.count("negative"),
        }
        
        summary[bgg_id] = {
            "count": len(items),
            "avg_quality": sum(quality_scores) / (10*len(quality_scores)) if quality_scores else None,
            "sentiments": sentiment_counts,
        }
    
    return summary

In [11]:
records = load_jsonl_dir("/Users/vishalsankarram/dsci558-project/game_forum")
agg = aggregate_by_bgg(records)
summary = summarize(agg)

print(summary)

{38726: {'count': 208, 'avg_quality': 0.7985576923076924, 'sentiments': {'positive': 181, 'neutral': 24, 'negative': 1}}, 19592: {'count': 57, 'avg_quality': 0.7929824561403509, 'sentiments': {'positive': 49, 'neutral': 8, 'negative': 0}}, 356080: {'count': 54, 'avg_quality': 0.7555555555555555, 'sentiments': {'positive': 42, 'neutral': 12, 'negative': 0}}, 16992: {'count': 362, 'avg_quality': 0.7718232044198895, 'sentiments': {'positive': 293, 'neutral': 66, 'negative': 3}}, 118: {'count': 553, 'avg_quality': 0.7835443037974683, 'sentiments': {'positive': 486, 'neutral': 63, 'negative': 2}}, 263918: {'count': 865, 'avg_quality': 0.789364161849711, 'sentiments': {'positive': 744, 'neutral': 115, 'negative': 6}}, 258779: {'count': 147, 'avg_quality': 0.7789115646258503, 'sentiments': {'positive': 124, 'neutral': 20, 'negative': 1}}, 233867: {'count': 676, 'avg_quality': 0.7866863905325444, 'sentiments': {'positive': 576, 'neutral': 98, 'negative': 2}}, 391752: {'count': 13, 'avg_quality

In [12]:
df["bgg_id"] = pd.to_numeric(df["bgg_id"], errors="coerce")
valid_ids = set(df["bgg_id"].dropna().astype(int))

In [13]:
for r in records:
    if r.get("bgg_id") is not None:
        try:
            r["bgg_id"] = int(r["bgg_id"])
        except:
            r["bgg_id"] = None

In [14]:
agg = aggregate_by_bgg(records)
summary = summarize(agg)

In [15]:
pos_ids = set()
neg_ids = set()

for bgg_id, data in summary.items():
    if bgg_id not in valid_ids:
        continue
    
    if data["avg_quality"] is not None and data["avg_quality"] < 0.65:
        neg_ids.add(bgg_id)
    else:
        pos_ids.add(bgg_id)

print(len(pos_ids), len(neg_ids))

7385 756


In [16]:
quality_map = {
    bgg_id: data["avg_quality"]
    for bgg_id, data in summary.items()
}

In [17]:
df["avg_quality"] = df["bgg_id"].map(quality_map)

In [18]:
df = df.dropna(subset=["avg_quality"])
df = df[df["stage_a_split"]=='train']
df = df.drop(columns=["stage_a_split"])
print(len(df))

7620


In [19]:
df = df.dropna(subset=["avg_quality"])

df["label"] = (df["avg_quality"] >= 0.65).astype(int) 

pos_df = df[df["label"] == 1]
neg_df = df[df["label"] == 0]

pos_sampled = pos_df.sample(n=len(neg_df), random_state=42)

balanced_df = pd.concat([pos_sampled, neg_df])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
balanced_df.drop(columns=["label"], inplace=True)
df=balanced_df

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

TARGET_COL = "avg_quality"

# keep only rows with target
df = df.dropna(subset=[TARGET_COL]).copy()

# use all columns except target and id as features
FEATURE_COLS = [c for c in df.columns if c not in [TARGET_COL, "bgg_id"]]

X = df[FEATURE_COLS]
y = df[TARGET_COL].astype(float).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(1180, 26) (296, 26) (1180,) (296,)


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(model, X_test, y_test, model_name, clip_min=0.0, clip_max=1.0):
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)

    # keep predictions in a sensible range
    y_pred = np.clip(y_pred, clip_min, clip_max)

    metrics_df = pd.DataFrame([{
        "model": model_name,
        "target": "avg_quality",
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
    }])

    out_df = pd.DataFrame({
        "true_avg_quality": y_test,
        "pred_avg_quality": y_pred,
        "model": model_name
    })

    return metrics_df, out_df

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

ridge_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=1.0))
])

cat_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", CatBoostRegressor(
        loss_function="RMSE",
        random_seed=42,
        verbose=0
    ))
])

lgbm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", LGBMRegressor(
        random_state=42
    ))
])

mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", MLPRegressor(
        hidden_layer_sizes=(128, 64),
        max_iter=500,
        random_state=42
    ))
])

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

In [23]:
ridge_model.fit(X_train, y_train)
ridge_metrics_df, ridge_pred_df = evaluate_model(
    ridge_model, X_test, y_test, model_name="Ridge"
)

cat_model.fit(X_train, y_train)
cat_metrics_df, cat_pred_df = evaluate_model(
    cat_model, X_test, y_test, model_name="CatBoost"
)

mlp_model.fit(X_train, y_train)
mlp_metrics_df, mlp_pred_df = evaluate_model(
    mlp_model, X_test, y_test, model_name="MLP"
)

lgbm_model.fit(X_train, y_train)
lgbm_metrics_df, lgbm_pred_df = evaluate_model(
    lgbm_model, X_test, y_test, model_name="LightGBM"
)

rf_model.fit(X_train, y_train)
rf_metrics_df, rf_pred_df = evaluate_model(
    rf_model, X_test, y_test, model_name="RandomForest"
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4899
[LightGBM] [Info] Number of data points in the train set: 1180, number of used features: 25
[LightGBM] [Info] Start training from score 0.655589


/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [24]:
results_df = pd.concat(
    [ridge_metrics_df, cat_metrics_df, mlp_metrics_df, lgbm_metrics_df, rf_metrics_df],
    ignore_index=True
)
# results_df = pd.concat(
#     [ridge_metrics_df, cat_metrics_df, mlp_metrics_df, lgbm_metrics_df],
#     ignore_index=True
# )

results_df = results_df.sort_values(["target", "mae"]).reset_index(drop=True)

summary_df = results_df[["model", "mae", "rmse", "r2"]].sort_values("mae").reset_index(drop=True)
print(summary_df)

          model       mae      rmse        r2
0  RandomForest  0.122906  0.155357  0.039384
1      CatBoost  0.123689  0.159960 -0.018393
2      LightGBM  0.125662  0.162910 -0.056297
3         Ridge  0.126548  0.157301  0.015187
4           MLP  0.144494  0.180627 -0.298547


In [25]:
from pathlib import Path
import joblib

MODELS_DIR = Path("value_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(ridge_model, MODELS_DIR / "ridge_model.joblib")
joblib.dump(cat_model, MODELS_DIR / "catboost_model.joblib")
joblib.dump(mlp_model, MODELS_DIR / "mlp_model.joblib")
joblib.dump(lgbm_model, MODELS_DIR / "lightgbm_model.joblib")
joblib.dump(rf_model, MODELS_DIR / "random_forest_model.joblib")

metadata = {
    "target_col": TARGET_COL,
    "feature_cols": FEATURE_COLS,
    "feature_count": len(FEATURE_COLS),
    "train_shape": X_train.shape,
    "test_shape": X_test.shape,
}

joblib.dump(metadata, MODELS_DIR / "metadata.joblib")
results_df.to_csv(MODELS_DIR / "model_results.csv", index=False)

In [26]:
ridge_model = joblib.load(MODELS_DIR / "ridge_model.joblib")
cat_model = joblib.load(MODELS_DIR / "catboost_model.joblib")
mlp_model = joblib.load(MODELS_DIR / "mlp_model.joblib")
lgbm_model = joblib.load(MODELS_DIR / "lightgbm_model.joblib")
rf_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")

metadata = joblib.load(MODELS_DIR / "metadata.joblib")